In [ ]:
#| default_exp bench

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, sys, os, ctypes, platform, subprocess
from fastcore.all import AttrDict

This page measures laghu against model2vec on public models: whether the output is identical, how much memory each uses, and how fast each encodes. The helpers here produce the table on the index page, and the sections after them record the experiments behind laghu's design.

## Measuring a process

`footprint()` reports the current process's memory in MB as `rss`, `private` and `peak`. `rss` is resident memory, which includes table pages that have been read from the mapped file. `private` is `phys_footprint` on macOS, the figure in Activity Monitor's Memory column, and anonymous resident memory (`RssAnon`) on Linux. The two are each platform's nearest measure of memory that belongs to the process alone, but they are not defined the same way.

`peak` is `lifetime_max_phys_footprint` on macOS and peak resident memory (`VmHWM`) on Linux, which includes mapped pages. Only compare figures taken on the same platform.

In [ ]:
#| export
def footprint():
    "The resident, private and peak memory of this process, in MB."
    if platform.system() == 'Darwin':
        buf = (ctypes.c_uint64 * 64)()   # rusage_info_v4 as uint64s: a 16-byte uuid, then resident_size at 8, phys_footprint 9, lifetime_max_phys_footprint 30
        ctypes.CDLL(None).proc_pid_rusage(ctypes.c_int(os.getpid()), 4, buf)
        return AttrDict(rss=buf[8] / 2**20, private=buf[9] / 2**20, peak=buf[30] / 2**20)
    kv = dict(l.split(':', 1) for l in open('/proc/self/status'))
    mb = lambda k: int(kv[k].split()[0]) / 1024
    return AttrDict(rss=mb('VmRSS'), private=mb('RssAnon'), peak=mb('VmHWM'))

In [ ]:
f = footprint(); assert 0 < f.private <= f.peak and f.rss > 0, f

`measure(path, how)` loads a model in a fresh Python process, with laghu or with model2vec, encodes `corpus(n)`, and returns the load and encode times and the memory figures. `check(path)` runs `same_as_model2vec` in a fresh process and adds what kind of model it is: tokenizer type, vocabulary size, dimensions, dtype and `normalize`. `row(path)` runs both and gives one line of the table, which `table(rows)` formats as markdown; `same(r)` says whether every check in a row passed. A fresh process for each run stops one measurement inheriting another's memory, and keeps a dozen large models from piling up in the notebook.

In [ ]:
#| export
_PROBE = '''
import json, time
from laghu.bench import footprint
from laghu.core import corpus, resolve
t = time.time()
if {how!r} == 'model2vec':
    from model2vec import StaticModel
    m = StaticModel.from_pretrained(str(resolve({path!r})), force_download=False)
else:
    from laghu import load
    m = load({path!r}, fallback=False)
tl = time.time() - t; f1 = footprint()
txts = corpus({n}); t = time.time(); m.encode(txts); te = time.time() - t; f2 = footprint()
print(json.dumps(dict(how={how!r}, load_s=tl, encode_s=te, texts=len(txts), private_load=f1.private, private=f2.private, rss=f2.rss, peak=f2.peak)))
'''

_CHECK = '''
import json
from laghu import load, same_as_model2vec
from laghu.core import corpus, model_files, resolve, find_vocab
m = load({path!r}, fallback=False); f = model_files(m.folder)
kind = json.loads(f.tok.read_text())['model'].get('type') or 'Unigram'
r = same_as_model2vec({path!r}, corpus({n}))
print(json.dumps(dict(r, kind=kind, vocab=len(m.embedding), dim=m.dim, dtype=m.embedding_dtype, normalize=bool(m.normalize),
    quantized=m.token_mapping is not None, weighted=m.weights is not None)))
'''

def _run(code):
    r = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True)
    if r.returncode: raise RuntimeError(r.stderr[-3000:])
    return AttrDict(json.loads(r.stdout.strip().splitlines()[-1]))

def measure(path, how='laghu', n=2000):
    "Memory in MB and time in seconds to load `path` with `how` ('laghu' or 'model2vec') and encode `corpus(n)`, in a new process."
    return _run(_PROBE.format(path=str(path), how=how, n=n))

def check(path, n=3000):
    "Run `same_as_model2vec(path, corpus(n))` in a new process and add the model's type and shape."
    return _run(_CHECK.format(path=str(path), n=n))

def row(path, n_check=3000, n_speed=2000):
    "One model's table row: what it is, whether it matches, and the memory and speed of both loaders."
    r = check(path, n_check); r.model = str(path)
    r.m2v, r.lg = measure(path, 'model2vec', n_speed), measure(path, 'laghu', n_speed)
    return r

In [ ]:
#| export
def same(r): return all(r[k] for k in ('ids', 'dtype', 'f32', 'f16', 'exact', 'seq', 'median', 'unk', 'tokens'))

def table(rows):
    "Format rows from `row` as a markdown table."
    hd = ('| model | tokenizer | laghu tokenizes with | vocab × dims | dtype | norm | identical | private MB, model2vec → laghu | peak MB, model2vec → laghu | texts/s, model2vec → laghu |\n'
          '|---|---|---|---|---|---|---|---|---|---|\n')
    def ln(r):
        extra = ' +map' * r.quantized + ' +weights' * r.weighted
        return (f"| {r.model.split('/')[-1]} | {r.kind} | {'Python Viterbi' if r.lean else 'tokenizers'} | {r.vocab:,} × {r.dim} | {r.dtype}{extra} | "
                f"{'on' if r.normalize else 'off'} | {'yes' if same(r) else 'NO'} | {r.m2v.private:,.0f} → {r.lg.private:,.0f} | {r.m2v.peak:,.0f} → {r.lg.peak:,.0f} | "
                f"{r.m2v.texts / r.m2v.encode_s:,.0f} → {r.lg.texts / r.lg.encode_s:,.0f} |")
    return hd + '\n'.join(map(ln, rows))

## Results

These measurements were taken on an M3 Max with macOS 26, Python 3.13, model2vec 0.9.0 and tokenizers 0.23.2. The equivalence checks use `corpus(3000)`, 4,726 texts, and compare the ids, the float32 and float16 vectors, the exact dtype and bits, `median_token_length`, the unknown id and the token list; `encode_as_sequence` is checked on 200 texts. Speed is `encode` over `corpus(2000)`, 3,226 texts, and private memory is taken after encoding. The cells marked `slow` download about 5 GB of models and run only with `nbdev-test --flags slow`. The comparison checks and full rows need model2vec (`laghu[fallback]`); a `measure` call for laghu alone does not.

`tokenizers` spreads a batch over all cores, while the Python Viterbi uses one, so for Unigram models laghu runs at between about a third of model2vec's rate and about the same on this machine. The other models run the same `tokenizers` code under both loaders; their 0.1-second timing samples vary by up to half between runs, so they show no real difference in speed. The rows from multilingual-e5-large-m2v down were measured from local copies of the repos on 2026-10-06, with the machine under load.

In [ ]:
#| slow
from IPython.display import Markdown
from laghu.core import try_resolve
import importlib.util
M2V = importlib.util.find_spec('model2vec') is not None   # without it there is nothing to compare with: skip
MODELS = [
    'minishlab/potion-base-2M', 'minishlab/potion-base-4M', 'minishlab/potion-base-8M', 'minishlab/potion-base-32M', 'minishlab/potion-retrieval-32M',
    'minishlab/potion-science-8M', 'minishlab/potion-code-16M', 'minishlab/potion-code-16M-v2', 'minishlab/potion-multilingual-128M',
    'minishlab/M2V_base_output', 'minishlab/M2V_multilingual_output', 'minishlab/M2V_base_glove', 'minishlab/M2V_base_glove_subword',
    'CISCai/jina-embeddings-v3-separation-distilled', 'nexus666/multilingual-e5-large-m2v', 'mikeee/m2v-gemma-embedding-300m',
    'Tokenade/potion-multilingual-128M-i8-tokenade', 'tss-deposium/m2v-qwen3-embedding-0.6b-1024d', 'OpenVoiceOS/ovos-m2v-intents-es-ES-bne-v5',
    'JanSchachtschabel/m2v-gte-256-edu', 'juampahc/bge-m3-m2v-256', 'blobbybob/potion-mxbai-micro', 'stephantulkens/NIFE-mxbai-embed-large-v1_model2vec',
    'OpenVoiceOS/ovos-m2v-intents-pt-PT-albertina-v5', 'giangndm/F2LLM-v2-330M-model2vec', 'amgix/static-retrieval-multilingual-69m-v1']
rows, failed = [], {}
for p in MODELS * M2V:
    if not try_resolve(p): continue
    try: rows.append(row(p))
    except Exception as e: failed[p] = repr(e)[-300:]
print(failed)
Markdown(table(rows))

In [ ]:
#| slow
assert not M2V or rows and not failed and all(same(r) for r in rows), [r.model for r in rows if not same(r)]

## A shared vocabulary, 2026-10-13

Before this change each process held potion-multilingual-128M's vocabulary as Python objects: a dict of 500,353 piece strings, an array of scores, and a table giving the longest piece for each pair of first letters. That came to about 80 MB of private memory per process, and VedicReader runs five such processes. Now the vocabulary is one file, built once and mapped by every process (see [unigram](unigram.html)). Two file formats were considered:

- Sorted UTF-8 pieces, or a hash table, with the ids and scores in parallel arrays, searched from Python. Every lookup would then be a search written in Python, competing with `dict.get`'s lookup in C, which already takes more than half of the Viterbi search's time.
- [marisa-trie](https://github.com/pytries/marisa-trie) 1.4.1 (MIT, its C++ core BSD-2-Clause or LGPL-2.1; wheels for CPython 3.9 to 3.14 on macOS, Linux x86_64 and aarch64, musl and Windows). `Trie.map` uses a trie stored in a buffer without copying it, and `iter_prefixes_with_ids` returns, in one call, every piece that starts at a given position.

marisa was chosen. The file is 7.6 MB for potion-multilingual-128M, and the ids do not change: 64,736 texts (60,000 lines, captions and meanings from VedicReader's library, `corpus(3000)` and ten made for this check: Vedic svaras, IAST, Tamil, emoji, a 10,000-character word, unknown scripts) give the same ids as the dict and as `tokenizers`, and the same embeddings, bit for bit. The figures below were measured on an M3 Max at a load average of 11 to 25, and each is the median of three fresh processes. USS is psutil's unique set size, the memory that belongs to one process alone, which on macOS only the process itself can read:

| potion-multilingual-128M | dict | shared file |
|---|---|---|
| load | 1.57 s | 0.05 s (1.73 s when it builds the file) |
| RSS after load and one encode | 149 MB | 73 MB |
| USS after load and one encode | 122 MB | 53 MB |
| USS each of 4 processes after 2,000 encodes | 127 MB | 53 MB |
| USS of the 4 together | 510 MB | 212 MB |
| Viterbi alone, 6,000 prose pre-tokens | 4,016/s | 9,651/s |
| the same, the 606 over 200 characters | 803/s | 2,157/s |
| the same, the 1,534 of 40 characters or fewer | 26,588/s | 32,956/s |
| one short pre-token, p50 / p95 | 58 / 113 µs | 37 / 52 µs |
| `encode`, 6,000 prose texts in one call, first pass | 1,820/s | 4,210/s |
| `encode`, the same in batches of 32, first pass | 2,766/s | 4,281/s |

The process that needs the file builds it in a child process, so the half-million strings of the build never enter its own memory. Limiting the prefix search to 12 or 24 characters, and extending it only where a longer piece could start, turned out slower than always searching up to the longest piece's length.

## Faster tokenizers? A spike, 2026-10-06

This experiment tried two Rust tokenizers in place of both the Python Viterbi and `tokenizers`, alongside cheaper changes to the Viterbi.

- [tokie](https://github.com/feyninc/tokie) 0.1.4: MIT or Apache-2.0, wheels for macOS arm64 and x86, Linux x86 and aarch64, Windows.
  BPE, WordPiece and Unigram from a tokenizer.json, on a double-array Aho-Corasick automaton. It claims to match `tokenizers` exactly.
- [gigatoken](https://github.com/marcelroed/gigatoken) 0.10.0: MIT, abi3 wheels for the same platforms. Byte-level BPE and
  SentencePiece BPE with byte fallback only: it refuses Unigram and WordPiece by name.

Each row comes from a fresh process. Memory is the private memory above that of an interpreter with laghu imported, measured after loading and again after tokenizing. Ids are checked against `tokenizers.encode_batch(add_special_tokens=False)` on `corpus(3000)`. Speed measures tokenizing alone over `corpus(2000)` (3,226 texts) and over 5,524 prose texts. The prose includes English notes with IAST, Sanskrit and Tamil verse and word-by-word glosses. Timings include the first pass and the best of two further passes. The machine's load average was about 16 throughout. Compare these rows with each other, not with the main table.

| model | tokenizer | ids differ (of 4,726) | MB after load | MB after encoding | corpus texts/s | prose texts/s | prose again, texts/s |
|---|---|---|---|---|---|---|---|
| potion-multilingual-128M (Unigram) | tokenizers, all cores | 0 | 706 | 791 | 44,961 | 75,200 | 117,332 |
| potion-multilingual-128M (Unigram) | tokenizers, 1 core | 0 | 705 | 775 | 16,947 | 14,142 | 18,429 |
| potion-multilingual-128M (Unigram) | tokie | 1 | 183 | 434 | 76,538 | 313,897 | 409,794 |
| potion-multilingual-128M (Unigram) | gigatoken | refuses: Unigram | | | | | |
| potion-multilingual-128M (Unigram) | Python Viterbi | 0 | 92 | 179 | 5,390 | 2,943 | 3,365 |
| potion-multilingual-128M (Unigram) | Python Viterbi + word cache | 0 | 92 | 214 | 5,394 | 3,216 | 24,766 |
| potion-multilingual-128M (Unigram) | Python Viterbi + cache, cut at ▁ | 1 | 96 | 187 | 6,612 | 12,640 | 21,277 |
| potion-multilingual-128M (Unigram) | Python Viterbi + cache, 4 processes | 0 | 97 | 110 (+182 × 4 workers) | 8,262 | 12,161 | 20,355 |
| jina-v3 distilled (Unigram) | tokenizers, all cores | 0 | 382 | 501 | 54,018 | 93,386 | 95,073 |
| jina-v3 distilled (Unigram) | tokenizers, 1 core | 0 | 380 | 477 | 24,385 | 18,019 | 20,797 |
| jina-v3 distilled (Unigram) | tokie | 43 | 144 | 388 | 74,681 | 259,734 | 385,673 |
| jina-v3 distilled (Unigram) | gigatoken | refuses: Unigram | | | | | |
| jina-v3 distilled (Unigram) | Python Viterbi | 0 | 48 | 172 | 6,157 | 6,734 | 6,591 |
| jina-v3 distilled (Unigram) | Python Viterbi + word cache | 0 | 45 | 177 | 6,757 | 11,044 | 27,315 |
| jina-v3 distilled (Unigram) | Python Viterbi + cache, cut at ▁ | 0 | 45 | 178 | 6,963 | 11,039 | 24,564 |
| jina-v3 distilled (Unigram) | Python Viterbi + cache, 4 processes | 0 | 46 | 62 (+157 × 4 workers) | 8,581 | 40,064 | 51,442 |
| gemma-300m m2v (Unigram, byte fallback) | tokenizers, all cores | 0 | 308 | 428 | 45,359 | 96,227 | 117,213 |
| gemma-300m m2v (Unigram, byte fallback) | tokenizers, 1 core | 0 | 309 | 400 | 18,254 | 15,377 | 18,927 |
| gemma-300m m2v (Unigram, byte fallback) | tokie | 1,822 | 95 | 266 | 115,058 | 385,430 | 455,435 |
| gemma-300m m2v (Unigram, byte fallback) | gigatoken | refuses: Unigram | | | | | |
| gemma-300m m2v (Unigram, byte fallback) | Python Viterbi | 0 | 45 | 140 | 4,998 | 3,756 | 4,631 |
| gemma-300m m2v (Unigram, byte fallback) | Python Viterbi + word cache | 0 | 43 | 146 | 5,316 | 4,415 | 27,386 |
| gemma-300m m2v (Unigram, byte fallback) | Python Viterbi + cache, cut at ▁ | refuses: piece spans ▁ | | | | | |
| gemma-300m m2v (Unigram, byte fallback) | Python Viterbi + cache, 4 processes | 0 | 43 | 67 (+146 × 4 workers) | 7,496 | 16,254 | 19,292 |
| potion-base-8M (WordPiece) | tokenizers, all cores | 0 | 19 | 32 | 64,956 | 90,916 | 133,482 |
| potion-base-8M (WordPiece) | tokenizers, 1 core | 0 | 20 | 24 | 26,680 | 26,283 | 26,995 |
| potion-base-8M (WordPiece) | tokie | 614 | 18 | 60 | 379,568 | 391,407 | 542,620 |
| potion-base-8M (WordPiece) | gigatoken | refuses: WordPiece | | | | | |
| NIFE-mxbai (WordPiece) | tokenizers, all cores | 0 | 45 | 98 | 70,691 | 146,824 | 161,753 |
| NIFE-mxbai (WordPiece) | tokenizers, 1 core | 0 | 46 | 48 | 28,335 | 27,703 | 26,909 |
| NIFE-mxbai (WordPiece) | tokie | 614 | 35 | 75 | 265,357 | 442,315 | 551,139 |
| NIFE-mxbai (WordPiece) | gigatoken | refuses: WordPiece | | | | | |
| static-retrieval-69m (BPE) | tokenizers, all cores | 0 | 179 | 436 | 39,165 | 135,455 | 156,584 |
| static-retrieval-69m (BPE) | tokenizers, 1 core | 0 | 175 | 376 | 19,600 | 27,725 | 28,010 |
| static-retrieval-69m (BPE) | tokie | 4,600 (1 panic) | 150 | 95 | 109 | 281,735 | 328,317 |
| static-retrieval-69m (BPE) | gigatoken | refuses: no single-byte entry for 0x00 | | | | | |
| F2LLM-v2-330M (BPE) | tokenizers, all cores | 0 | 92 | 383 | 22,818 | 90,630 | 127,481 |
| F2LLM-v2-330M (BPE) | tokenizers, 1 core | 0 | 92 | 339 | 14,913 | 17,507 | 17,701 |
| F2LLM-v2-330M (BPE) | tokie | 4,587 (1 panic) | 86 | 128 | 272,607 | 370,923 | 377,640 |
| F2LLM-v2-330M (BPE) | gigatoken | 4,020 | 98 | 98 | 1,056 | 278,809 | 419,339 |
| albertina (BPE, ByteLevel) | tokenizers, all cores | 0 | 27 | 105 | 23,834 | 92,308 | 139,109 |
| albertina (BPE, ByteLevel) | tokenizers, 1 core | 0 | 30 | 42 | 16,301 | 20,785 | 21,587 |
| albertina (BPE, ByteLevel) | tokie | 1 (1 panic) | 42 | 51 | 159 | 269,176 | 313,442 |
| albertina (BPE, ByteLevel) | gigatoken | 0 | 23 | 70 | 14,625 | 352,702 | 458,499 |

What the table shows:

- tokie is fast and small but not exact. It gives different ids on every model: a stray `▁` for a whitespace-only text (jina-v3),
  the other order of an exact tie (`′` `′′` against `′′` `′`, potion-multilingual), real ids where `tokenizers` gives `[UNK]` (Thai,
  Armenian; WordPiece), and different ids on almost every text of two BPE models. On BPE it panics on a long run of `नम`. Inside
  `encode_batch` the panic hangs the call. A batch of 256 holding the 200,000-character string takes 30 s, against 0.3 s for that string
  alone. Its memory grows from 180 to 430 MB on potion-multilingual as it encodes.
- gigatoken is exact on one of the three BPE models (albertina, a GPT-2-style byte-level BPE), at about 3.8 times `tokenizers` on
  prose with less memory. On F2LLM (Qwen's tokenizer: NFC, a custom split regex, `add_prefix_space`) it gives different ids on 4,020
  texts, and it refuses to load static-retrieval. laghu tokenizes BPE with `tokenizers`, whose BPE tables are small (27 to 179 MB here).
  gigatoken could only be an opt-in for models it is checked on, never the default.
- A cache of segmented words cannot change any id, because `viterbi`'s result depends only on the pre-token. It helps where the pre-tokenizer
  splits words. potion-multilingual-128M's Metaspace has `split: false`, which makes each text one pre-token. There the cache helps only a
  text seen before.
- Cutting the text at each `▁` before the Viterbi search, to give potion-multilingual word-sized pre-tokens, is not exact. No piece contains a `▁`
  except as its first character, but each cut piece of text sums its scores from zero instead of from the start of the text. One exact tie
  (the primes above) then breaks the other way. One text in 4,726 differs, and laghu does not cut.
- Four worker processes give 1.5 to 6 times the rate, but each worker holds 146 to 182 MB of its own, the very cost laghu exists
  to avoid. laghu does not use workers; a caller who wants a pool can run one.
- numpy was not tried. Profiling shows more than half the time in `dict.get` on substrings (3.6 million lookups for 2,000 texts),
  which numpy cannot speed up.

The word cache that laghu adopted holds up to `WORD_CACHE` pre-tokens of at most `WORD_LEN` characters per `Unigram`. Checks against `tokenizers` found no differing ids on the 4,946 corpus and added-token texts, or on the prose, for the Unigram tokenizers tested. The table compares prose texts/s without the cache, with the cache on a first pass and on a second pass. Each measurement is one run on the same loaded machine:

| tokenizer | pre-tokens | no cache | cache, first pass | cache, again |
|---|---|---|---|---|
| jina-embeddings-v3-separation-distilled | words (WhitespaceSplit, Metaspace) | 6,288 | 10,031 | 25,082 |
| bge-m3-m2v-256 | words | 5,764 | 10,872 | 29,602 |
| multilingual-e5-large-m2v | words | 6,424 | 10,616 | 19,941 |
| potion-multilingual-128M | the whole text (Metaspace, `split: false`) | 2,702 | 2,537 | 2,614 |
| potion-multilingual-128M-i8-tokenade | the whole text | 2,670 | 2,684 | 2,714 |
| m2v-gemma-embedding-300m | the whole text | 3,681 | 3,657 | 3,691 |
| m2v-qwen3-embedding-0.6b-1024d | the whole text | 3,557 | 3,530 | 3,536 |
| ovos-m2v-intents-es-ES-bne-v5 | the whole text (ByteLevel) | 5,006 | 5,199 | 4,951 |

The cache holds about 32,000 words after the prose and adds about 5 MB.